# Универсальный ноутбук обучения прогноза количества операций

Пайплайн: загрузка → фильтрация → извлечение операций → feature engineering → сохранение Excel → обучение ML → прогноз → дашборд.

Поддерживает: CatBoost, LightGBM, XGBoost, Ridge. Режим «топ-N худших» (фильтр по resist_id).

## 1. Конфиг

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Добавляем корень проекта в путь для импорта build_metrics_dashboard_simple
sys.path.insert(0, str(Path("../").resolve()))

# --- Конфиг ---
RAW_DIR = Path("../data/raw")
RAW_FILES = [
    RAW_DIR / "data-2022.csv",
    RAW_DIR / "data-2023.csv",
    RAW_DIR / "data-2024.csv",
    RAW_DIR / "data-2025.csv",
]

FILTER_YEAR = 2025

# Топ-N худших resist_id (из дашборда). Пустой список = все подразделения.
TOP_N_WORST_RESIST_IDS = []  # Пример: [123, 456, 789] — только эти модели численности

OUTPUT_EXCEL = Path("../data/processed/training_data_features.xlsx")
OUTPUT_DASHBOARD_DIR = Path("../data/processed")

## 2. Загрузка и фильтрация (из build_dataframe_for_dashboard)

In [ ]:
PREDICT_COMMENT = "first_predict_value_record"
EXCLUDED_RESIST_IDS = [
    25, 26, 27, 29, 54, 57, 61, 63, 66, 68, 80, 81, 82, 83, 100, 101, 102, 103,
    104, 105, 106, 109, 112, 113, 115, 116, 119, 120, 121, 122, 123, 125, 126,
    129, 131, 132, 134, 135, 136, 137, 138, 140, 141, 143, 144, 146, 147, 148,
    149, 150, 151, 152, 153, 154, 156, 157, 158, 162, 165, 169, 170, 171, 172,
    173, 174, 175, 176, 177, 178, 179, 181, 196, 51, 65, 139,
]

def to_bool(series):
    return series.astype(str).str.lower().isin(["true", "t", "1"])

def to_bool_value(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return None
    return str(value).strip().lower() in ["true", "t", "1"]

def parse_division_title(div_json):
    if pd.isna(div_json):
        return None
    try:
        obj = json.loads(div_json)
    except Exception:
        return None
    return obj.get("name")

def parse_calculate_param(calc_json):
    if pd.isna(calc_json):
        return {}
    try:
        return json.loads(calc_json)
    except Exception:
        return {}

def get_calc_value(calc_obj, key):
    val = calc_obj.get(key)
    if isinstance(val, dict):
        return val.get("value")
    return val

# Загрузка CSV
dfs = []
for p in RAW_FILES:
    if p.exists():
        dfs.append(pd.read_csv(p, dtype=str))
if not dfs:
    raise FileNotFoundError("Нет CSV-файлов для загрузки.")

df_raw = pd.concat(dfs, ignore_index=True)
df_raw["period"] = pd.to_datetime(df_raw["period"], errors="coerce")
df_raw["resist_id"] = pd.to_numeric(df_raw["resist_id"], errors="coerce").astype("Int64")

# Фильтрация (clean_and_filter)
if "cancel_approve" in df_raw.columns:
    df_raw["cancel_approve"] = to_bool(df_raw["cancel_approve"])
else:
    df_raw["cancel_approve"] = False

if "comment" in df_raw.columns:
    df_raw["comment"] = df_raw["comment"].fillna("")
    mask_predict = df_raw["comment"] == PREDICT_COMMENT
else:
    mask_predict = pd.Series(False, index=df_raw.index)

mask_ok = ~df_raw["cancel_approve"] & ~mask_predict & ~df_raw["resist_id"].isin(EXCLUDED_RESIST_IDS)
df_ok = df_raw[mask_ok].copy()

print(f"После фильтрации: {len(df_ok)} строк")

## 3. Извлечение операций + workDayMonth_2

In [ ]:
records = []
for _, row in df_ok.iterrows():
    calc_obj = parse_calculate_param(row.get("calculate_parameter"))
    workday = get_calc_value(calc_obj, "workDayMonth_2")
    op_raw = row.get("operation_parameter")
    if pd.isna(op_raw):
        continue
    try:
        op_data = json.loads(op_raw)
    except Exception:
        continue
    period = row.get("period")
    if pd.isna(period):
        continue
    for op_id, payload in op_data.items():
        if not isinstance(payload, dict):
            continue
        active = to_bool_value(payload.get("operation_active", payload.get("active")))
        if active is False:
            continue
        records.append({
            "resist_id": row.get("resist_id"),
            "division_title": parse_division_title(row.get("division_name")),
            "id": row.get("id"),
            "month": int(period.month),
            "year": int(period.year),
            "operation_id": op_id,
            "fact_value": payload.get("fact_value"),
            "fact_predict": payload.get("fact_predict"),
            "workDayMonth_2": workday,
            "calculate_minute": payload.get("calculate_minute"),
        })

df = pd.DataFrame(records)
df["fact_value"] = pd.to_numeric(df["fact_value"], errors="coerce")
df["fact_predict"] = pd.to_numeric(df["fact_predict"], errors="coerce")
df["workDayMonth_2"] = pd.to_numeric(df["workDayMonth_2"], errors="coerce")
df["calculate_minute"] = pd.to_numeric(df["calculate_minute"], errors="coerce")

# Фильтр по топ-N худшим (если задан)
if TOP_N_WORST_RESIST_IDS:
    df = df[df["resist_id"].isin(TOP_N_WORST_RESIST_IDS)].copy()
    print(f"Фильтр топ-N: {len(df)} строк")

print(f"Операций: {len(df)}")

## 4. Производные колонки и фильтрация нулей

In [ ]:
df["coefficient_fulltime_staff"] = df["calculate_minute"] / 60 / 148.3
df["fact_fts"] = df["fact_value"] * df["coefficient_fulltime_staff"]
df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1), errors="coerce").dt.date

df = df.dropna(subset=["fact_value"])

# Фильтрация начальных нулевых месяцев (filter_leading_zeros)
df["_grp"] = df["resist_id"].astype(str) + "|" + df["division_title"].astype(str) + "|" + df["operation_id"].astype(str)
yr_mask = df["year"] == FILTER_YEAR
drop_idx = set()
for g in df["_grp"].unique():
    grp = df[(df["_grp"] == g) & yr_mask].sort_values("month")
    for idx in grp.index:
        fv, fp = df.at[idx, "fact_value"], df.at[idx, "fact_predict"]
        if (pd.isna(fv) or float(fv) == 0) and (pd.isna(fp) or float(fp) == 0):
            drop_idx.add(idx)
        else:
            break
df = df.drop(index=list(drop_idx)).drop(columns=["_grp"]).reset_index(drop=True)

df = df.sort_values(["operation_id", "year", "month"]).reset_index(drop=True)
print(f"После очистки: {len(df)} строк")

## 5. Feature engineering

In [ ]:
group = df.groupby("operation_id")
df["lag_1"] = group["fact_value"].shift(1)
df["lag_12"] = group["fact_value"].shift(12)
df["roll_mean_3"] = group["fact_value"].transform(lambda s: s.shift(1).rolling(3, min_periods=3).mean())
df["roll_mean_6"] = group["fact_value"].transform(lambda s: s.shift(1).rolling(6, min_periods=6).mean())
df["trend"] = df["lag_1"] / df["roll_mean_3"]
df["trend"] = df["trend"].replace([np.inf, -np.inf], 0)

num_cols = df.select_dtypes(include=["number"]).columns
df[num_cols] = df[num_cols].fillna(0)
df = df.infer_objects(copy=False)

df["mape_fact_predict"] = np.where(
    df["fact_value"] == 0,
    np.where(df["fact_predict"] == 0, 0.0, 100.0),
    np.clip((1 - (df["fact_value"] - df["fact_predict"]).abs() / df["fact_value"].abs()) * 100, 0, 100)
)
df["abs_error"] = (df["fact_value"] - df["fact_predict"]).abs()

print(df.head())

## 6. Сохранение датасета в Excel

In [ ]:
OUTPUT_EXCEL.parent.mkdir(parents=True, exist_ok=True)
df.to_excel(OUTPUT_EXCEL, index=False)
print(f"Датасет сохранён: {OUTPUT_EXCEL}")

## 7. model_factory и обучение

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error

def model_factory(name: str):
    """Возвращает модель по имени. CatBoost, LightGBM, XGBoost требуют установки."""
    if name == "catboost":
        from catboost import CatBoostRegressor
        return CatBoostRegressor(iterations=1000, depth=6, learning_rate=0.1, loss_function="RMSE",
                                random_seed=42, verbose=100, early_stopping_rounds=50)
    if name == "lightgbm":
        import lightgbm as lgb
        return lgb.LGBMRegressor(n_estimators=1000, max_depth=6, learning_rate=0.1,
                                 random_state=42, verbose=100, early_stopping_rounds=50)
    if name == "xgboost":
        from xgboost import XGBRegressor
        return XGBRegressor(n_estimators=1000, max_depth=6, learning_rate=0.1,
                            random_state=42, verbosity=0)
    if name == "ridge":
        return Ridge(alpha=1.0, random_state=42)
    raise ValueError(f"Неизвестная модель: {name}")

CAT_COLS = ["operation_id", "division_title", "resist_id"]
NUM_COLS = ["month", "year", "workDayMonth_2", "calculate_minute", "coefficient_fulltime_staff",
            "lag_1", "lag_12", "roll_mean_3", "roll_mean_6", "trend"]
FEATURE_COLS = CAT_COLS + NUM_COLS
TARGET_COL = "fact_value"

In [ ]:
MODEL_NAME = "catboost"  # catboost | lightgbm | xgboost | ridge

df_train = df.copy()
df_train[CAT_COLS] = df_train[CAT_COLS].astype(str)

last_year = df_train["year"].max()
train_df = df_train[df_train["year"] < last_year]
test_df = df_train[df_train["year"] == last_year]

X_train = train_df[FEATURE_COLS].copy()
X_test = test_df[FEATURE_COLS].copy()
y_train = train_df[TARGET_COL]
y_test = test_df[TARGET_COL]

model = model_factory(MODEL_NAME)

if MODEL_NAME == "catboost":
    cat_idx = [FEATURE_COLS.index(c) for c in CAT_COLS]
    train_weight = train_df["calculate_minute"].fillna(train_df["calculate_minute"].mean())
    model.fit(X_train, y_train, cat_features=cat_idx, eval_set=(X_test, y_test),
              sample_weight=train_weight)
elif MODEL_NAME == "lightgbm":
    for c in CAT_COLS:
        le = LabelEncoder()
        X_train[c] = le.fit_transform(X_train[c].astype(str))
        X_test[c] = le.transform(X_test[c].astype(str))
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)])
elif MODEL_NAME == "xgboost":
    for c in CAT_COLS:
        le = LabelEncoder()
        X_train[c] = le.fit_transform(X_train[c].astype(str))
        X_test[c] = le.transform(X_test[c].astype(str))
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
else:
    for c in CAT_COLS:
        le = LabelEncoder()
        X_train[c] = le.fit_transform(X_train[c].astype(str))
        X_test[c] = le.transform(X_test[c].astype(str))
    model.fit(X_train, y_train)

pred = np.maximum(model.predict(X_test), 0)
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
print(f"Test MAE: {mae:.2f}, RMSE: {rmse:.2f}")

## 8. Формирование датафрейма с прогнозом и вызов дашборда

In [ ]:
forecast_col = f"predict_{MODEL_NAME}"
df_dash = test_df.copy()
df_dash[forecast_col] = pred
df_dash["fact_predict"] = df_dash[forecast_col]  # для совместимости с колонками дашборда

# Колонки для дашборда
dash_cols = ["resist_id", "division_title", "operation_id", "date", "month", "year",
             "fact_value", "fact_predict", "fact_fts"]
df_dash = df_dash[[c for c in dash_cols if c in df_dash.columns]]

# Сохраняем для дашборда
dashboard_input = OUTPUT_DASHBOARD_DIR / f"forecast_input_{MODEL_NAME}.xlsx"
df_dash.to_excel(dashboard_input, index=False)

from build_metrics_dashboard_simple import build_dashboard

out_path = OUTPUT_DASHBOARD_DIR / f"forecast_dashboard_{MODEL_NAME}.xlsx"
build_dashboard(dashboard_input, output_path=out_path, forecast_col="fact_predict")
print(f"Дашборд: {out_path}")